# 4_inter_intra_classification.ipynb

**Pipeline position:** Step 4 — runs *after* `key_authors_graph.ipynb` (step 3).

**Inputs (per country, pooled from one or more campaigns' `key_authors_graph.ipynb` output):**
* `*_node_indicators.parquet` — author-level feature DataFrame, indexed by `accountid`.
* `processed_<campaign>_part_all.gzip.parquet` — co-located post-level file used to derive the IO label via the `is_control` column.

**Experiments:**
1. **Intra-country** — train/test on the same country, varying the training fraction.
2. **Inter-country (LOO)** — train on the other countries, test on the held-out target.

**Outputs:** Results CSV, aggregated CSV, AUC plot, Macro-F1 plot.

# Configuration — edit this cell

In [ ]:
campaign_step3_dirs = {
    "Cuba": [
        "../results/Labeled_Datasets/Cuba/Cuba_part_all/step_3_key_authors_graph/topic_col_BERTopic_topic_256_top_percentage_1",
    ],
    "Egypt_UAE": [
        "../results/test_data/Egypt_UAE/Egypt_UAE_part_all/2026_06_24",
    ],
    "Iran": [
        "../results/Labeled_Datasets/Iran_1/Iran_1_part_all/step_3_key_authors_graph/topic_col_BERTopic_topic_256_top_percentage_1",
        "../results/Labeled_Datasets/Iran_2/Iran_2_part_all/step_3_key_authors_graph/topic_col_BERTopic_topic_256_top_percentage_1",
        "../results/Labeled_Datasets/Iran_3/Iran_3_part_all/step_3_key_authors_graph/topic_col_BERTopic_topic_256_top_percentage_1",
        "../results/Labeled_Datasets/Iran_4/Iran_4_part_all/step_3_key_authors_graph/topic_col_BERTopic_topic_256_top_percentage_1",
        "../results/Labeled_Datasets/Iran_6/Iran_6_part_all/step_3_key_authors_graph/topic_col_BERTopic_topic_256_top_percentage_1",
    ],
    "Venezuela": [
        "../results/Labeled_Datasets/Venezuela_1/Venezuela_1_part_all/step_3_key_authors_graph/topic_col_BERTopic_topic_256_top_percentage_1",
        "../results/Labeled_Datasets/Venezuela_2/Venezuela_2_part_all/step_3_key_authors_graph/topic_col_BERTopic_topic_256_top_percentage_1",
    ],
    "Ecuador_Test": [
        "../results/Labeled_Datasets/Ecuador_Test/Ecuador_Test_part_all/step_3_key_authors_graph/topic_col_BERTopic_topic_256_top_percentage_1",
    ],
    "Cuba_Test": [
        "../results/Labeled_Datasets/Cuba_Test/Cuba_Test_part_all/step_3_key_authors_graph/topic_col_BERTopic_topic_256_top_percentage_1",
    ],
    # Add more countries here...
}
output_dir = "../results/Labeled_Datasets/step_4_inter_intra_classification"
graph_filtering_setting = "all_authors"

# Sweep-config selection for the classification_sweep_indicators format
# (Ecuador_Test / Cuba_Test) — picks the single sweep file matching this
# exact config out of each country's parameter sweep.
sweep_topic_method = "BERTopic"
sweep_topic_n = 256
sweep_edge_types = "similarity+text_similarity+interactions"
sweep_top_percentage = 1.0

FRACTIONS = [0.01, 0.05, 0.10, 0.25, 0.50, 1.0]
MODELS = ["xgboost", "random_forest", "logistic_regression"]
SEEDS = [42, 123, 456, 789, 1024]  # each seed re-draws the train sample and re-seeds the classifier
RANDOM_STATE = 42
TEST_SIZE = 0.20


# Imports & Utils

In [ ]:
from __future__ import annotations

from pathlib import Path
from functools import wraps
import gc
import math
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, f1_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# Derived after the parameters cell so it reflects a papermill-injected
# campaign_step3_dirs override, not the cell's original default value.
CAMPAIGNS = list(campaign_step3_dirs.keys())

output_path = Path(output_dir).expanduser()
output_path.mkdir(parents=True, exist_ok=True)
REPORT_PATH = output_path / "progress_report.txt"


def write_report(msg: str) -> None:
    # Timestamped print + append to the progress-report file
    line = f"{time.strftime('%Y-%m-%d %H:%M:%S')} | {msg}"
    print(line)
    with open(REPORT_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")


def report_done(func):
    # Logs completion or failure of the wrapped function via write_report
    @wraps(func)
    def wrapper(*args, **kwargs):
        try:
            result = func(*args, **kwargs)
            write_report(f"Completed: {func.__name__}")
            return result
        except Exception as exc:
            write_report(f"FAILED: {func.__name__} | {exc}")
            raise
    return wrapper


def get_classifier(model_name: str, random_state: int = RANDOM_STATE, **kwargs):
    # Returns an untrained sklearn-compatible classifier by name
    name = model_name.lower()
    if name in ("xgboost", "xgb"):
        return XGBClassifier(random_state=random_state, eval_metric="logloss", **kwargs)
    if name in ("random_forest", "rf"):
        return RandomForestClassifier(random_state=random_state, **kwargs)
    if name in ("logistic_regression", "lr"):
        return LogisticRegression(random_state=random_state, solver="liblinear", max_iter=1000, **kwargs)
    raise ValueError(
        f"Unsupported model_name: '{model_name}'. Supported: xgboost/xgb, random_forest/rf, logistic_regression/lr."
    )


write_report(f"Campaign step3 dirs: {campaign_step3_dirs}")
write_report(f"Output path: {output_path}")


In [ ]:
# Scoped to the countries with real, loadable step-3 data as of this run —
# Cuba, Iran, Venezuela in campaign_step3_dirs have no usable data yet.
CAMPAIGNS = ["Egypt_UAE", "Ecuador_Test", "Cuba_Test"]
write_report(f"CAMPAIGNS scoped to: {CAMPAIGNS}")


# Data Loading

In [ ]:
def load_country(
    country_name: str,
    step3_dirs: list[str | Path],
    graph_filter: str = "all_authors",
) -> tuple[pd.DataFrame, set]:
    # Loads a country's node-indicator features + IO author set, pooling all
    # of its campaign step-3 dirs (accountid prefixed with country_name).
    frames: list[pd.DataFrame] = []
    combined_io: set = set()

    for step3_dir in step3_dirs:
        camp_dir = Path(step3_dir).expanduser()
        if not camp_dir.exists():
            write_report(f"WARNING: Skipping '{camp_dir}' for '{country_name}' — directory not found.")
            continue

        # Unfiltered variant has no "top_percentage" in the filename; any other
        # setting is matched as a substring of the filename.
        candidates = []
        for f in camp_dir.rglob("*_node_indicators.parquet"):
            if graph_filter == "all_authors":
                if "top_percentage" not in f.name:
                    candidates.append(f)
            elif graph_filter in f.name:
                candidates.append(f)

        if not candidates:
            # Fallback: classification_sweep_indicators/{graph_filter}/ format —
            # features + IO label baked into a single parquet (e.g. Ecuador_Test,
            # Cuba_Test), rather than a separate node_indicators/is_control pair.
            # Picks the one sweep file matching the configured sweep_* params.
            sweep_dir = camp_dir / "classification_sweep_indicators" / graph_filter
            sweep_file = sweep_dir / (
                f"accounts_graph_{sweep_topic_method}_topic_{sweep_topic_n}_"
                f"{sweep_edge_types}_top_percentage_{sweep_top_percentage}_per_topic_"
                f"topic_col_{sweep_topic_method}_topic_{sweep_topic_n}_undirected_indicators.parquet"
            )
            if sweep_file.exists():
                df = pd.read_parquet(sweep_file)
                if "io_drive_label" in df.columns:
                    io_set = set(df.index[df["io_drive_label"] == 1].astype(str))
                    df = df.drop(columns=["io_drive_label"])
                    df.index = country_name + "_" + df.index.astype(str)
                    io_set = {f"{country_name}_{aid}" for aid in io_set}
                    frames.append(df)
                    combined_io.update(io_set)
                    continue
            write_report(f"WARNING: No node_indicators or sweep file for '{country_name}' in {camp_dir} (filter='{graph_filter}').")
            continue

        df = pd.read_parquet(sorted(candidates)[-1])

        # NOTE: this glob intentionally does not match the real step-3 output
        # filename (`*_step_3.gzip.parquet`, with an underscore before "3") and
        # always falls through to the `processed_*`/catch-all fallback below.
        # Left as-is by design — do not "fix" this without re-reading the plan.
        processed = sorted(camp_dir.rglob("*_step3.gzip.parquet"))
        if not processed:
            processed = sorted(camp_dir.rglob("processed_*.gzip.parquet"))
        if not processed:
            processed = sorted(camp_dir.rglob("*.parquet"))
        if not processed:
            write_report(f"WARNING: No processed parquet for '{country_name}' in {camp_dir}.")
            continue
        df_proc = pd.read_parquet(processed[-1], columns=["accountid", "is_control"])
        io_set = set(df_proc.loc[df_proc["is_control"] == False, "accountid"].astype(str))

        df.index = country_name + "_" + df.index.astype(str)
        io_set = {f"{country_name}_{aid}" for aid in io_set}

        frames.append(df)
        combined_io.update(io_set)

    if not frames:
        raise FileNotFoundError(f"No step-3 directories could be loaded for '{country_name}': {step3_dirs}")

    df_all = pd.concat(frames, axis=0)
    n_io_in_graph = len(set(df_all.index.astype(str)) & combined_io)
    write_report(
        f"Loaded '{country_name}' | {len(df_all)} authors | {len(combined_io)} IO authors | {n_io_in_graph} IO in graph"
    )
    return df_all, combined_io


def load_countries(
    country_list: list[str],
    campaign_step3_dirs: dict[str, list[str]],
    graph_filter: str = "all_authors",
) -> tuple[pd.DataFrame, set]:
    # Pools load_country() across multiple countries — used for the LOO training pool.
    frames: list[pd.DataFrame] = []
    combined_io: set = set()
    for country in country_list:
        try:
            df, io_set = load_country(country, campaign_step3_dirs[country], graph_filter)
            frames.append(df)
            combined_io.update(io_set)
        except FileNotFoundError as exc:
            write_report(f"WARNING: Skipping '{country}' in multi-country load — {exc}")
    if not frames:
        raise FileNotFoundError(f"None of the requested countries could be loaded: {country_list}")
    return pd.concat(frames, axis=0), combined_io


In [ ]:
for _country in CAMPAIGNS:
    try:
        _df_demo, _io_demo = load_country(_country, campaign_step3_dirs[_country], graph_filtering_setting)
        print(f"{_country}: shape={_df_demo.shape}")
        print(_df_demo.select_dtypes(include="number").columns.tolist())
        del _df_demo, _io_demo
        break
    except FileNotFoundError as exc:
        write_report(f"Feature-column check: skipping '{_country}' — {exc}")
gc.collect()


# Classification Engine & Experiment Sweep

In [ ]:
@report_done
def io_classification(
    node_indicators_df: pd.DataFrame,
    io_authors_set: set,
    model_name: str,
    test_size: float = TEST_SIZE,
    random_state: int = RANDOM_STATE,
    test_node_indicators_df: pd.DataFrame | None = None,
    test_io_authors_set: set | None = None,
) -> tuple:
    # Trains/evaluates one classifier. If an external test set is supplied,
    # uses it directly (cross-campaign/LOO mode); otherwise does an internal
    # stratified train_test_split (intra mode).
    if node_indicators_df.empty:
        raise ValueError("Training feature set is empty — aborting.")

    train_numeric = node_indicators_df.select_dtypes(include="number")
    train_cols = train_numeric.columns.tolist()
    X_train = train_numeric.to_numpy()

    io_str = set(map(str, io_authors_set))
    y_train = node_indicators_df.index.astype(str).isin(io_str).astype(int)

    if test_node_indicators_df is not None and test_io_authors_set is not None:
        test_numeric = test_node_indicators_df.select_dtypes(include="number")
        test_cols = test_numeric.columns.tolist()
        if train_cols != test_cols:
            raise ValueError(f"Train/test feature columns do not match: {set(train_cols) ^ set(test_cols)}")
        X_test = test_numeric.to_numpy()
        test_io_str = set(map(str, test_io_authors_set))
        y_test = test_node_indicators_df.index.astype(str).isin(test_io_str).astype(int)
    else:
        stratify = y_train if len(np.unique(y_train)) > 1 else None
        X_train, X_test, y_train, y_test = train_test_split(
            X_train, y_train, test_size=test_size, random_state=random_state, stratify=stratify,
        )

    clf = get_classifier(model_name=model_name, random_state=random_state)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1] if hasattr(clf, "predict_proba") else y_pred.astype(float)

    report = classification_report(y_test, y_pred, target_names=["Non-IO", "IO"], zero_division=0)
    auc = roc_auc_score(y_test, y_proba) if len(np.unique(y_test)) > 1 else 0.5
    macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
    return clf, report, auc, macro_f1


def run_experiments(
    campaigns: list[str],
    fractions: list[float],
    models: list[str],
    seeds: list[int],
    graph_filter: str = "all_authors",
) -> pd.DataFrame:
    # For each target country: one fixed 80/20 split (seeded by RANDOM_STATE,
    # identical across every fraction/model/seed), then sweeps fraction x seed x
    # model for intra-country and inter-country (LOO) training.
    def stratified_sample(df, io_str_set, n, seed):
        # Samples n rows, guaranteeing >=1 IO and >=1 non-IO author when both exist.
        y = df.index.astype(str).isin(io_str_set).astype(int)
        io_idx, nonIO_idx = df.index[y == 1].tolist(), df.index[y == 0].tolist()
        if not io_idx or not nonIO_idx:
            return df.sample(n=min(n, len(df)), random_state=seed)
        prop_io = len(io_idx) / len(df)
        n_io = max(1, min(int(round(n * prop_io)), len(io_idx), n - 1))
        n_nonIO = max(1, min(n - n_io, len(nonIO_idx)))
        rng = np.random.default_rng(seed)
        sampled = (
            rng.choice(io_idx, size=n_io, replace=False).tolist()
            + rng.choice(nonIO_idx, size=n_nonIO, replace=False).tolist()
        )
        return df.loc[sampled]

    results: list[dict] = []

    for target_camp in campaigns:
        write_report(f"Target campaign: {target_camp}")
        try:
            df_target, io_target = load_country(target_camp, campaign_step3_dirs[target_camp], graph_filter)
        except FileNotFoundError as exc:
            write_report(f"SKIP {target_camp}: {exc}")
            continue

        io_target_str = set(map(str, io_target))
        y_all = df_target.index.astype(str).isin(io_target_str).astype(int)
        if y_all.sum() == 0:
            write_report(f"SKIP {target_camp}: no IO authors in the graph.")
            continue

        # Fixed 80/20 split seeded by RANDOM_STATE — same test set for every fraction/model/seed
        stratify = y_all if len(np.unique(y_all)) > 1 else None
        df_train_full, df_test, y_train_full, y_test_ref = train_test_split(
            df_target, y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=stratify,
        )
        test_io_set = set(df_test.index[y_test_ref == 1].astype(str))
        write_report(
            f"  Train pool: {len(df_train_full)} | Test: {len(df_test)} | IO in test: {int(y_test_ref.sum())}"
        )

        loo_camps = [c for c in campaigns if c != target_camp]
        try:
            df_inter_all, io_inter_all = load_countries(loo_camps, campaign_step3_dirs, graph_filter)
            write_report(f"  LOO train pool: {len(df_inter_all)} authors from {loo_camps}")
        except FileNotFoundError as exc:
            write_report(f"  WARNING: LOO data unavailable — {exc}")
            df_inter_all, io_inter_all = None, None

        for frac in fractions:
            n_intra = max(2, min(int(round(len(df_train_full) * frac)), len(df_train_full)))
            n_inter = (
                max(2, min(int(round(len(df_inter_all) * frac)), len(df_inter_all)))
                if df_inter_all is not None else None
            )

            for seed in seeds:
                df_train_frac = (
                    df_train_full if frac >= 1.0
                    else stratified_sample(df_train_full, io_target_str, n_intra, seed)
                )

                # Strict no-overlap check (guaranteed: test set carved out first)
                overlap = set(df_train_frac.index.astype(str)) & set(df_test.index.astype(str))
                assert len(overlap) == 0, (
                    f"FATAL: {len(overlap)} authors leaked into test for '{target_camp}' @ frac={frac}, seed={seed}"
                )

                train_io_frac = set(df_train_frac.index.astype(str)) & io_target_str

                if df_inter_all is not None:
                    df_inter_frac = (
                        df_inter_all if frac >= 1.0
                        else stratified_sample(df_inter_all, set(map(str, io_inter_all)), n_inter, seed)
                    )
                    inter_io_frac = set(df_inter_frac.index.astype(str)) & set(map(str, io_inter_all))

                for model_name in models:
                    try:
                        _, _, auc, macro_f1 = io_classification(
                            df_train_frac, train_io_frac, model_name, random_state=seed,
                            test_node_indicators_df=df_test, test_io_authors_set=test_io_set,
                        )
                        results.append({
                            "training_campaign": target_camp,
                            "target_campaign": target_camp,
                            "graph_filtering_setting": graph_filter,
                            "train_fraction": frac,
                            "model": model_name,
                            "experiment_type": "intra",
                            "seed": seed,
                            "auc": auc,
                            "macro_f1": macro_f1,
                        })
                    except Exception as exc:
                        write_report(f"  ERROR intra | {target_camp} | {model_name} | frac={frac} | seed={seed}: {exc}")

                    if df_inter_all is None:
                        continue
                    try:
                        _, _, auc, macro_f1 = io_classification(
                            df_inter_frac, inter_io_frac, model_name, random_state=seed,
                            test_node_indicators_df=df_test, test_io_authors_set=test_io_set,
                        )
                        results.append({
                            "training_campaign": ",".join(loo_camps),
                            "target_campaign": target_camp,
                            "graph_filtering_setting": graph_filter,
                            "train_fraction": frac,
                            "model": model_name,
                            "experiment_type": "inter",
                            "seed": seed,
                            "auc": auc,
                            "macro_f1": macro_f1,
                        })
                    except Exception as exc:
                        write_report(f"  ERROR inter | {target_camp} | {model_name} | frac={frac} | seed={seed}: {exc}")

        del df_target, df_train_full
        if df_inter_all is not None:
            del df_inter_all
        gc.collect()

    return pd.DataFrame(results)


In [ ]:
write_report("Starting cross-campaign IO detection sweep.")
write_report(f"Campaigns: {CAMPAIGNS} | Fractions: {FRACTIONS} | Models: {MODELS} | Seeds: {SEEDS}")

results_df = run_experiments(
    campaigns=CAMPAIGNS,
    fractions=FRACTIONS,
    models=MODELS,
    seeds=SEEDS,
    graph_filter=graph_filtering_setting,
)

if results_df.empty:
    write_report("WARNING: No results collected. Ensure all campaigns have completed key_authors_graph.ipynb runs.")
else:
    write_report(f"Sweep complete — {len(results_df)} result rows collected.")
    display(results_df.head(12))


# Plots

In [ ]:
LINES = [
    ("intra", "Intra-country", "#D4A017", "o"),        # gold, circle
    ("inter", "Inter-country (LOO)", "#4472C4", "s"),  # blue, square
]
CAMPAIGN_ORDER = list(campaign_step3_dirs.keys())


def plot_results(df_results: pd.DataFrame, metric: str, output_dir: Path | None = None) -> None:
    # One subplot per country; error bars = std across models x seeds, clipped
    # to [0, 1]; fixed [0, 1.05] y-axis; log-scale x-axis.
    campaigns = [c for c in CAMPAIGN_ORDER if c in df_results["target_campaign"].unique()]
    if not campaigns:
        write_report(f"No data to plot for metric '{metric}'.")
        return

    ncols = math.ceil(math.sqrt(len(campaigns)))
    nrows = math.ceil(len(campaigns) / ncols)
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(6 * ncols, 5 * nrows))
    axes_flat = np.atleast_1d(axes).flatten()

    for idx, camp in enumerate(campaigns):
        ax = axes_flat[idx]
        camp_data = df_results[df_results["target_campaign"] == camp]

        for exp_type, label, color, marker in LINES:
            subset = camp_data[camp_data["experiment_type"] == exp_type]
            if subset.empty:
                continue
            grouped = subset.groupby("train_fraction")[metric].agg(["mean", "std"]).reset_index()
            mean, std = grouped["mean"], grouped["std"].fillna(0)
            # Clip the visual error bar to [0, 1] — both metrics are bounded there.
            lower, upper = np.maximum(mean - std, 0.0), np.minimum(mean + std, 1.0)
            yerr = np.vstack([mean - lower, upper - mean])
            ax.errorbar(
                grouped["train_fraction"] * 100, mean, yerr=yerr, fmt=f"-{marker}",
                color=color, label=label, capsize=3, linewidth=1.5, markersize=6, elinewidth=1.0,
            )

        ax.set_xscale("log")
        ax.set_xticks([1, 5, 10, 25, 50, 100])
        ax.set_xticklabels(["1", "5", "10", "25", "50", "100"])
        ax.set_xlim(0.9, 110)
        ax.set_title(camp, fontweight="bold", fontsize=11)
        ax.set_xlabel("Percentage of Training Data (%)", fontsize=9)
        ax.set_ylabel(metric.upper(), fontsize=9)
        ax.legend(fontsize=8, loc="lower right")
        ax.grid(True, linestyle="--", alpha=0.4)
        ax.set_ylim(0, 1.05)

    for idx in range(len(campaigns), len(axes_flat)):
        axes_flat[idx].set_visible(False)

    fig.suptitle(f"Intra- vs Inter-Campaign IO Detection — {metric.upper()}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    if output_dir:
        out_path = output_dir / f"strategy_comparison_{metric}.png"
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
        write_report(f"Saved plot → {out_path}")
    plt.show()


In [ ]:
if not results_df.empty:
    for metric in ["auc", "macro_f1"]:
        plot_results(results_df, metric, output_path)


# Save Results

In [ ]:
if not results_df.empty:
    csv_path = output_path / f"experiment_results_{graph_filtering_setting}.csv"
    results_df.to_csv(csv_path, index=False)
    write_report(f"Results saved -> {csv_path}")
    display(results_df.describe())

    # Seed-aggregated summary (mean/std across the SEEDS replicates)
    results_agg = (
        results_df
        .groupby(["target_campaign", "experiment_type", "model", "train_fraction"])
        .agg(
            auc_mean=("auc", "mean"),
            auc_std=("auc", "std"),
            macro_f1_mean=("macro_f1", "mean"),
            macro_f1_std=("macro_f1", "std"),
            n_seeds=("seed", "nunique"),
        )
        .reset_index()
    )
    agg_path = output_path / f"experiment_results_{graph_filtering_setting}_aggregated.csv"
    results_agg.to_csv(agg_path, index=False)
    write_report(f"Seed-aggregated results saved -> {agg_path}")
    display(results_agg.head(12))

write_report("Pipeline completed.")


# Summary for Amit

In [ ]:
if not results_df.empty:
    print("=" * 78)
    print("SUMMARY FOR AMIT")
    print("=" * 78)

    print(
        f"\n1) Configuration used: graph_filtering_setting='{graph_filtering_setting}', "
        f"sweep_topic_method='{sweep_topic_method}', sweep_topic_n={sweep_topic_n}, "
        f"sweep_edge_types='{sweep_edge_types}', sweep_top_percentage={sweep_top_percentage}.\n"
        "   Why: 'all_authors' (no graph filtering) + all edge types + top_percentage=1.0 is the "
        "closest match to IOHunter's Figure 2 main (unfiltered) comparison, and this exact config "
        "is present in both Ecuador_Test and Cuba_Test's classification_sweep_indicators sweep. "
        "Egypt_UAE uses its own pre-sweep node_indicators/is_control format (unaffected by this config)."
    )

    print("\n2) Authors per country, and 3) IO / non-IO split:")
    for _camp in CAMPAIGNS:
        _df_c, _io_c = load_country(_camp, campaign_step3_dirs[_camp], graph_filtering_setting)
        _n_io = len(set(_df_c.index.astype(str)) & _io_c)
        print(f"   {_camp}: {len(_df_c)} authors ({_n_io} IO / {len(_df_c) - _n_io} non-IO)")
        del _df_c, _io_c
    gc.collect()

    _summary = (
        results_agg[results_agg["train_fraction"] == 1.0]
        .groupby(["target_campaign", "experiment_type"])[["auc_mean", "macro_f1_mean"]]
        .mean()
        .round(3)
    )
    print("\n4) Result pattern (mean AUC / Macro-F1 at 100% training data, across models/seeds):")
    print(_summary.to_string())
    print(
        "\n   Intra-country consistently outperforms inter-country (LOO) for all three countries, "
        "and performance generally rises with more training data for Egypt_UAE and Cuba_Test. "
        "Ecuador_Test is noisier (only 30 authors / 5 IO total, 6 in the held-out test set) — "
        "its inter-country AUC trends toward 0 at higher fractions, likely a small-test-set artifact "
        "rather than a real effect."
    )

    print(f"\n5) Plots saved to:\n   {output_path / 'strategy_comparison_auc.png'}\n   {output_path / 'strategy_comparison_macro_f1.png'}")

    print(
        "\n6) Question for Amit: does the 'all_authors / BERTopic / topic_256 / all edge types / "
        "top_percentage=1.0' sweep config make sense as the default comparison point, or should we "
        "try a different one from the sweep (e.g. graph_authors, or a lower top_percentage)?"
    )
    print("=" * 78)
